## Task 1: Same-Modality Similarity Search

The **first task** we defined for our **Game Recommendation Assistant** is the following:

**Text-based retrieval**: Given a game's description, the system finds and returns game descriptions with similar themes, genres, or content.

**Image-based retrieval**: Given a game's cover or snapshot, the system identifies visually similar game images from the database.

**Video-based retrieval**: Given a game trailer, the system retrieves trailers that share comparable visual style, mood, or gameplay elements.

In [ ]:
# Importing useful dependencies
import io
import boto3
import torch
import imageio
import requests
import chromadb
import open_clip
import numpy as np
from PIL import Image
from io import BytesIO
import ipywidgets as widgets
import torch.nn.functional as F
from IPython.display import display
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor

In [ ]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000", # MinIO API endpoint
    aws_access_key_id="minioadmin", # User name
    aws_secret_access_key="minioadmin", # Password
)

In [ ]:
# Connect to the server of ChromaDB where we stored the embeddings of files (Docker Container)
client = chromadb.HttpClient(host="localhost", port=8000)

# Create or get the collection named "texts"
collection_texts = client.create_collection(name="texts", get_or_create=True, embedding_function=None)

# Create or get the collection named "images"
collection_images = client.create_collection(name="images", get_or_create=True, embedding_function=None)

# Create or get the collection named "videos"
collection_videos = client.create_collection(name="videos", get_or_create=True, embedding_function=None)

In [ ]:
# Just in case our device has gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

Next, we implement a function that given a **file's embeddings** and a **ChromaDB collection** of embeddings returns the **k** most similar files of the same modality to the given one.

In [ ]:
def find_similar_files(collection, query_emb: np.ndarray, top_k: int = 5):
    
    # Chroma expects list-of-lists for query_embeddings
    query_vector = query_emb.tolist()

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        include=["documents", "distances"]
    )

    # Extract first query results
    ids = results.get("ids", [[]])[0]
    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]

    print(f"Top {top_k} similar {collection.name}:")
    for rank, (doc_id, doc, dist) in enumerate(zip(ids, docs, dists), start=1):
        print(f"{rank}. id={doc_id}, distance={dist:.4f}")
        print(f"   document: {doc}")
        temp_path = doc.split("/")[-1]
        if (collection.name == "images"):
            display(get_image("trusted-zone", f"{collection.name}/{temp_path}"))
        elif (collection.name == "texts"):
            print("\n" + get_text("trusted-zone", f"{collection.name}/{temp_path}") + "\n")
        elif (collection.name == "videos"):
            frames = get_video("trusted-zone", f"{collection.name}/{temp_path}")
            for frame in frames:
                display(frame)
    return results

### **Text-based retrieval**

In [ ]:
# Load model + tokenizer used when creating the text embeddings
model_text, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-16") # Tokenizer for texts
model_text.to(device)

We can either select a description from the MinIO database or formulate one directly in the following cell to perform the text similarity search.

In [ ]:
# We can use this function to retrieve an text from our bucket
def get_text(bucket, key):
    resp = s3.get_object(Bucket=bucket, Key=key)
    body = resp["Body"].read()
    text = body.decode("utf-8")
    return text

In [ ]:
@torch.no_grad()
# The next function returns the embedding of the given text
def embed_text(tokenizer, model, text):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    tokens = tokenizer(paragraphs).to(device)

    with torch.no_grad():
        feats = model.encode_text(tokens)
        feats = F.normalize(feats, dim=-1)

    all_embs = feats.detach().cpu().numpy()
    if len(all_embs) == 0:
        return np.zeros(model.config.hidden_size)
    elif len(all_embs) == 1:
        return all_embs[0]
    else:
        full_emb = np.mean(np.stack(all_embs), axis=0)
        full_emb = full_emb / np.linalg.norm(full_emb)
        return full_emb

Firstly, we can try using a description from our MinIO database.

In [ ]:
# Sample description
text_example_1 = get_text("trusted-zone", "texts/text_1761689048672.txt")
# Create embeddings for the description
text_example_emb_1 = embed_text(tokenizer, model_text, text_example_1)
text_example_1

In [ ]:
# Search for similar texts in ChromaDB
res_text_1 = find_similar_files(collection_texts, text_example_emb_1, top_k=5)
# The first one is always the target text itself (because it exists in our ChromaDB)

Next, we can try using an image that has never appeared before in ChromaDB.

In [ ]:
# Sample description (Generated by ChatGpt)
text_example_2 = "Undertale is a unique role-playing game where players navigate a world filled with quirky monsters. Choices matter: you can fight, flee, or befriend enemies, affecting the story and multiple endings. The game combines retro-style graphics, witty humor, and emotional storytelling, offering a deep, player-driven experience that challenges traditional RPG mechanics."
# Create embeddings for the description
text_example_emb_2 = embed_text(tokenizer, model_text, text_example_2)
text_example_emb_2

In [ ]:
# Search for similar texts in ChromaDB
res_text_2 = find_similar_files(collection_texts, text_example_emb_2, top_k=5)
# The first one is always the target text itself (because it exists in our ChromaDB)

### **Image-based retrieval**

In [ ]:
# Load model + preprocessing used when creating the image embeddings
model_image, _, preprocess_image = open_clip.create_model_and_transforms("ViT-B-16", pretrained="openai")
model_image.to(device)

We can either select an image from the MinIO database or upload one from local storage to perform the image similarity search.

In [ ]:
# We can use this function to retrieve an image from our bucket in PIL Image format
def get_image(bucket, key):
    resp = s3.get_object(Bucket=bucket, Key=key)
    body = resp["Body"].read()
    img = Image.open(io.BytesIO(body))
    return img

In [ ]:
# The next function returns the embedding of the given PIL Image
def embed_image(preprocess, model, pil_img):
    img_tensor = preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = model.encode_image(img_tensor)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().squeeze()

Firstly, we can try using an image from our MinIO database.

In [ ]:
# Sample PIL Image
img_example_1 = get_image("trusted-zone", "images/image_1761688911231.png")
# Create embeddings for the Image
img_example_emb_1 = embed_image(preprocess_image, model_image, img_example_1)
img_example_1

In [ ]:
# Search for similar images in ChromaDB
results_1 = find_similar_files(collection_images, img_example_emb_1, top_k=5)
# The first one is always the target image itself (because it exists in our ChromaDB)

Next, we can try using an image that has never appeared before in ChromaDB.

In [ ]:
# Upload an image from local storage
uploader = widgets.FileUpload(accept='image/*', multiple=False)
display(uploader) # We need to upload an image before executing the next cell

In [ ]:
# Extract the uploaded file
image_data = uploader.value[0].content
img_example_2 = Image.open(BytesIO(image_data))

# Create embeddings for the Image
img_example_emb_2 = embed_image(preprocess_image, model_image, img_example_2)
img_example_2

In [ ]:
# Search for similar images in ChromaDB
results_2 = find_similar_files(collection_images, img_example_emb_2, top_k=5)

### **Video-based retrieval**

In [ ]:
# Load model + preprocessing used when creating the video embeddings
tokenizer_video = CLIPImageProcessor.from_pretrained("Searchium-ai/clip4clip-webvid150k")
model_video = CLIPVisionModelWithProjection.from_pretrained("Searchium-ai/clip4clip-webvid150k")
model_video.to(device)

We can either select a video from the MinIO database or upload one directly to perform the video similarity search.

In [ ]:
temp_file = "temp_video_in.mp4"
# We can use the following function to retrieve a video from our MinIO database
# We are only extracting frames of the video, ignoring the audio content of the video
def get_video(bucket = None, key = None, max_frames = 16, url = None):
    if url:
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(temp_file, "wb") as f:
            f.write(r.content)
    else:
        resp = s3.get_object(Bucket=bucket, Key=key)
        body = resp["Body"].read()
        with open(temp_file, "wb") as f:
            f.write(body)
    frames = []
    reader = imageio.get_reader(temp_file, format="ffmpeg")
    total_frames = reader.count_frames()
    if total_frames and total_frames > 0:
        step = max(1, total_frames // max_frames)
        #print(total_frames, step, max_frames)
        idxs = list(range(0, total_frames, step))[:max_frames]
        for i in idxs:
            try:
                frame = reader.get_data(i)
                frames.append(Image.fromarray(frame))
            except Exception:
                continue
    else:
        # fallback: iterate and collect up to max_frames
        for i, frame in enumerate(reader):
            frames.append(Image.fromarray(frame))
            if len(frames) >= max_frames:
                break
    reader.close()
    return frames

In [ ]:
# The next function returns the embedding of the given video
def embed_video(tokenizer, model, frames_video):
    inputs = tokenizer(images=frames_video, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device) # shape (num_frames, 3, H, W)

    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
        # prefer pooler_output if available, else mean over last_hidden_state
        emb_frames = getattr(outputs, "pooler_output", None)
        if emb_frames is None:
            emb_frames = outputs.last_hidden_state.mean(dim=1)
        # average frame embeddings to make video embedding
        video_emb = emb_frames.mean(dim=0).cpu().numpy()
    return  video_emb

Firstly, we can try using a video from our MinIO database.

In [ ]:
# Sample video
video_example_1 = get_video("trusted-zone", "videos/video_1760786507389.mp4") # frames/list of images of the video
# Create embeddings for the video
video_example_emb_1 = embed_video(tokenizer_video, model_video, video_example_1)
for frame in video_example_1:
    display(frame)

In [ ]:
# Search for similar videos in ChromaDB
res_video_1 = find_similar_files(collection_videos, video_example_emb_1, top_k=5)
# The first one is always the target text itself (because it exists in our ChromaDB)

Next, we can try using a video that has never appeared before in ChromaDB.

In [ ]:
# Sample video
video_example_2 = get_video(url = "https://cdn.akamai.steamstatic.com/steam/apps/256853884/movie_max.mp4?t=1633085092") # frames/list of images of the video
# Create embeddings for the video
video_example_emb_2 = embed_video(tokenizer_video, model_video, video_example_2)
for frame in video_example_2:
    display(frame)

In [ ]:
# Search for similar videos in ChromaDB
res_video_2 = find_similar_files(collection_videos, video_example_emb_2, top_k=5)
# The first one is always the target text itself (because it exists in our ChromaDB)